# Stable Diffusion XL with Hyper-SD LoRA

This notebook generates images using **Stability AI's SDXL 1.0** with **Hyper-SD LoRA** for fast inference.

## Edge Cases & Error Handling Covered:
- ❌ CUDA unavailable → falls back to CPU (with warning)
- ❌ Disk space low → checks before model download
- ❌ HuggingFace rate limit / network error → retry with backoff
- ❌ Model file corrupted → delete and re-download
- ❌ OOM (out of memory) → clear cache and retry with smaller settings
- ❌ Invalid prompt → validate and sanitize
- ❌ Output directory missing → auto-create
- ❌ User interrupts kernel → graceful cleanup
- ❌ Package import failure → informative error with fix instructions

In [ ]:
# Cell 1: Install dependencies with comprehensive error handling
import subprocess
import sys

try:
    %pip install --quiet --upgrade diffusers transformers accelerate mediapy peft
    print("[OK] All packages installed successfully.")
except Exception as e:
    print(f"[ERROR] Package installation failed: {e}")
    print("[INFO] Attempting individual package installation...")
    packages = ['diffusers', 'transformers', 'accelerate', 'mediapy', 'peft']
    for pkg in packages:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', pkg])
            print(f"  [OK] {pkg} installed.")
        except Exception as e2:
            print(f"  [FAIL] {pkg} could not be installed: {e2}")
            print(f"  [FIX]  Run: !pip install {pkg}")
    raise RuntimeError("Some packages failed to install. See above for details.")

In [ ]:
# Cell 2: Imports with per-module error handling and diagnostics
import importlib
import os
import random
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', category=FutureWarning)

REQUIRED_MODULES = {
    'mediapy': 'mediapy',
    'torch': 'torch',
    'diffusers': 'diffusers',
    'transformers': 'transformers',
    'huggingface_hub': 'huggingface_hub',
}

loaded = {}
for name, import_name in REQUIRED_MODULES.items():
    try:
        loaded[name] = importlib.import_module(import_name)
        print(f"[OK] Loaded: {name}")
    except ImportError as e:
        print(f"[ERROR] Could not import '{name}': {e}")
        print(f"[FIX]   Run: !pip install {name}")
    except Exception as e:
        print(f"[ERROR] Unexpected error importing '{name}': {e}")

if 'diffusers' in loaded:
    from diffusers import DiffusionPipeline, TCDScheduler
if 'huggingface_hub' in loaded:
    from huggingface_hub import hf_hub_download, HfFolder, HfHubHTTPError

media = loaded.get('mediapy')
torch = loaded.get('torch')

if not all([media, torch, 'diffusers' in loaded, 'huggingface_hub' in loaded]):
    raise RuntimeError("Critical dependencies missing. Cannot continue.")

print("\n[OK] All imports resolved.")

In [ ]:
# Cell 3: Device detection with CPU fallback
def detect_device():
    """Detect best available device with comprehensive fallback logic."""
    if not torch.cuda.is_available():
        print("[WARN] CUDA is NOT available. Falling back to CPU.")
        print("[WARN] CPU inference will be VERY slow (minutes per image).")
        print("[FIX]  In Colab: Runtime > Change runtime type > T4 GPU")
        return 'cpu'

    try:
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
        print(f"[OK] CUDA available. Device: {gpu_name} ({gpu_mem:.1f} GB VRAM)")

        if gpu_mem < 4:
            print("[WARN] Less than 4 GB VRAM detected. SDXL may OOM.")
            print("[INFO] Will use memory optimizations.")
            return 'cuda', {'low_mem': True}
        return 'cuda', {'low_mem': False}
    except Exception as e:
        print(f"[WARN] GPU detection error: {e}. Falling back to CPU.")
        return 'cpu'

device_info = detect_device()
if isinstance(device_info, tuple):
    device, opts = device_info
else:
    device = device_info
    opts = {}

print(f"[INFO] Using device: {device}")
if opts.get('low_mem'):
    print("[INFO] Low memory mode enabled.")

In [ ]:
# Cell 4: Configuration with input validation
import math

# --- Inference steps ---
try:
    num_inference_steps = int(os.environ.get('NUM_STEPS', 12))
    num_inference_steps = max(1, min(50, num_inference_steps))  # clamp [1, 50]
except (ValueError, TypeError):
    print("[WARN] Invalid NUM_STEPS env var. Using default: 12")
    num_inference_steps = 12

base_model_id = os.environ.get('BASE_MODEL', 'stabilityai/stable-diffusion-xl-base-1.0')
repo_name = "ByteDance/Hyper-SD"
plural = "s" if num_inference_steps > 1 else ""
ckpt_name = f"Hyper-SDXL-{num_inference_steps}step{plural}-CFG-lora.safetensors"

print(f"[CONFIG] Model: {base_model_id}")
print(f"[CONFIG] LoRA:   {repo_name}/{ckpt_name}")
print(f"[CONFIG] Steps:  {num_inference_steps}")
print(f"[CONFIG] Device: {device}")

In [ ]:
# Cell 5: Disk space check before downloading models
def get_free_disk_gb(path='.'):
    """Get free disk space in GB. Returns None if unavailable."""
    try:
        st = os.statvfs(path)
        return (st.f_frsize * st.f_bavail) / (1024**3)
    except (AttributeError, OSError):
        # statvfs not available on all platforms (e.g., Windows)
        return None

# SDXL model ~7 GB, LoRA ~100 MB, plus cache
MIN_REQUIRED_GB = 15
free_gb = get_free_disk_gb()

if free_gb is not None:
    print(f"[INFO] Free disk space: {free_gb:.1f} GB")
    if free_gb < MIN_REQUIRED_GB:
        print(f"[WARN] Low disk space ({free_gb:.1f} GB). Need >{MIN_REQUIRED_GB} GB.")
        print("[WARN] Model download may fail. Consider freeing space.")
else:
    print("[INFO] Could not check disk space (statvfs not available). Proceeding anyway.")

In [ ]:
# Cell 6: Load pipeline with comprehensive error handling
MAX_RETRIES = 2
RETRY_DELAY = 5  # seconds

def load_pipeline():
    """Load the SDXL pipeline with retries and fallback."""
    for attempt in range(1, MAX_RETRIES + 2):  # attempt 1, then retry up to MAX_RETRIES
        try:
            print(f"[INFO] Loading model (attempt {attempt})...")

            dtype = torch.float16 if device == 'cuda' else torch.float32
            variant = 'fp16' if device == 'cuda' else None

            pipe = DiffusionPipeline.from_pretrained(
                base_model_id,
                torch_dtype=dtype,
                variant=variant,
                use_safetensors=True
            )

            if opts.get('low_mem'):
                pipe.enable_model_cpu_offload()
                pipe.enable_vae_slicing()
                pipe.enable_vae_tiling()
                print("[INFO] Enabled memory optimizations (CPU offload, VAE slicing/tiling).")
            else:
                pipe = pipe.to(device)

            # Load LoRA with retry logic
            print(f"[INFO] Downloading LoRA weights: {ckpt_name}...")
            lora_path = hf_hub_download(
                repo_name,
                ckpt_name,
                resume_download=True,
                force_download=False
            )

            pipe.load_lora_weights(lora_path)
            pipe.fuse_lora()
            pipe.scheduler = TCDScheduler.from_config(pipe.scheduler.config)

            # Warm up with a tiny dummy pass
            if device == 'cuda':
                _ = pipe(
                    prompt='test',
                    num_inference_steps=1,
                    guidance_scale=1.0,
                    width=64,
                    height=64,
                )
                print("[OK] Pipeline warm-up completed.")

            return pipe

        except Exception as e:
            err_str = str(e).lower()
            print(f"[ERROR] Attempt {attempt} failed: {e}")

            if '401' in err_str or 'unauthorized' in err_str:
                print("[FIX]  Authentication error. Log in with: huggingface-cli login")
                print("[FIX]  Or use a model that doesn't require authentication.")
                raise  # No point retrying auth errors

            if 'gated' in err_str or 'access' in err_str:
                print("[FIX]  This model requires access approval.")
                print("[FIX]  Visit: https://huggingface.co/" + base_model_id)
                raise

            if 'out of memory' in err_str or 'cuda out of memory' in err_str:
                print("[INFO] OOM detected. Clearing cache and retrying with memory opts...")
                torch.cuda.empty_cache()
                opts['low_mem'] = True
                continue  # Retry with memory optimizations

            if 'corrupted' in err_str or 'checksum' in err_str or 'magic' in err_str:
                print("[INFO] Model file may be corrupted. Re-downloading...")
                if attempt < MAX_RETRIES + 1:
                    time.sleep(RETRY_DELAY)
                    continue

            if attempt < MAX_RETRIES + 1:
                print(f"[INFO] Retrying in {RETRY_DELAY}s...")
                time.sleep(RETRY_DELAY)
            else:
                print("[FATAL] All retries exhausted. Could not load pipeline.")
                raise

    return None


pipe = load_pipeline()
print("\n[OK] Pipeline ready for inference.")

In [ ]:
# Cell 7: Generate image with comprehensive edge case handling

# --- Memory management ---
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
if device == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# --- Prompt from env var or default ---
prompt = os.environ.get('PROMPT', "a photo of Pikachu fine dining with a view to the Eiffel Tower")

# Validate prompt
if not prompt or not prompt.strip():
    print("[WARN] Empty prompt provided. Using fallback prompt.")
    prompt = "a beautiful landscape with mountains and sunset"
prompt = prompt.strip()
if len(prompt) > 1000:
    print("[WARN] Prompt exceeds 1000 chars. Truncating.")
    prompt = prompt[:1000]

# --- Seed with validation ---
seed_str = os.environ.get('SEED', '')
if seed_str:
    try:
        seed = int(seed_str)
        seed = max(0, min(2**31 - 1, seed))
    except (ValueError, TypeError):
        print(f"[WARN] Invalid SEED '{seed_str}'. Using random seed.")
        seed = random.randint(0, 2**31 - 1)
else:
    seed = random.randint(0, 2**31 - 1)

# --- Guidance scale with validation ---
gs_str = os.environ.get('GUIDANCE_SCALE', '')
if gs_str:
    try:
        guidance_scale = float(gs_str)
        guidance_scale = max(1.0, min(20.0, guidance_scale))  # clamp [1, 20]
    except (ValueError, TypeError):
        print(f"[WARN] Invalid GUIDANCE_SCALE '{gs_str}'. Using default: 5.0")
        guidance_scale = 5.0
else:
    guidance_scale = 5.0

# --- Eta with validation ---
eta_str = os.environ.get('ETA', '')
if eta_str:
    try:
        eta = float(eta_str)
        eta = max(0.0, min(1.0, eta))  # clamp [0, 1]
    except (ValueError, TypeError):
        print(f"[WARN] Invalid ETA '{eta_str}'. Using default: 0.5")
        eta = 0.5
else:
    eta = 0.5

# --- Output path ---
output_dir = os.environ.get('OUTPUT_DIR', '.')
output_path = Path(output_dir)
output_path.mkdir(parents=True, exist_ok=True)
output_file = output_path / f"output_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.jpg"

print(f"\n[PROMPT]  {prompt}")
print(f"[SEED]    {seed}")
print(f"[GUIDE]   {guidance_scale}")
print(f"[ETA]     {eta}")
print(f"[STEPS]   {num_inference_steps}")
print(f"[OUTPUT]  {output_file}")
print()

# --- Generate with error handling ---
try:
    # Check that pipeline was loaded
    if pipe is None:
        raise RuntimeError("Pipeline not loaded. Run the previous cell first.")

    generator_kwargs = {}
    if device == 'cuda':
        generator = torch.Generator(device).manual_seed(seed)
        generator_kwargs['generator'] = generator

    images = pipe(
        prompt=prompt,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        eta=eta,
        **generator_kwargs,
    ).images

    if not images or len(images) == 0:
        raise RuntimeError("Pipeline returned no images.")

    print(f"[OK] Generated {len(images)} image(s).")
    print(f"Prompt: {prompt}")
    print(f"Seed:   {seed}")

    # Display the image
    media.show_images(images)

    # Save with overwrite confirmation
    if output_file.exists():
        print(f"[INFO] Overwriting existing file: {output_file}")
    images[0].save(str(output_file))
    print(f"[OK] Image saved to: {output_file}")

    # Also save with seed in filename for reproducibility
    seed_file = output_path / f'output_seed{seed}.jpg'
    images[0].save(str(seed_file))
    print(f"[OK] Backup saved to: {seed_file}")

except RuntimeError as e:
    err_str = str(e).lower()
    if 'out of memory' in err_str or 'cuda' in err_str:
        print(f"[ERROR] Out of memory during inference.")
        print("[FIX]  Try lowering num_inference_steps, guidance_scale, or using CPU.")
        print("[FIX]  Or restart the runtime: Runtime > Restart runtime")
        torch.cuda.empty_cache()
    else:
        print(f"[ERROR] Inference failed: {e}")
except KeyboardInterrupt:
    print("\n[WARN] User interrupted inference. Cleaning up...")
    if device == 'cuda':
        torch.cuda.empty_cache()
except Exception as e:
    print(f"[ERROR] Unexpected error during inference: {e}")
    import traceback
    traceback.print_exc()
finally:
    # Always attempt cleanup
    if device == 'cuda':
        try:
            before = torch.cuda.memory_allocated() / 1024**3
            torch.cuda.empty_cache()
            after = torch.cuda.memory_allocated() / 1024**3
            print(f"[CLEANUP] GPU memory: {before:.2f} GB → {after:.2f} GB")
        except Exception:
            pass
    print("[DONE] Cell execution complete.")